In [289]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from scipy import stats
from xgboost import XGBRegressor
random_state = 45
target = 'saleprice'

**1---Data Inspection**

In [290]:
df_raw = pd.read_csv('data/train.csv')  #reading the training dataset
df_test = pd.read_csv('data/test.csv')   #reading the test dataset
print('---raw data---')
display(df_raw.head())
print('---raw data shape---')
display(df_raw.shape)
print('---data info---')
display(df_raw.info())
print('---data description---')
display(df_raw.describe().T)

print('---test data---')
display(df_test.head())
print('---test data shape---')
display(df_test.shape)
print('---test data info---')
display(df_test.info())
print('---test data description---')
display(df_test.describe().T)


---raw data---


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


---raw data shape---


(1460, 81)

---data info---
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  Ove

...

(1459, 80)

---test data info---
<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   str    
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   str    
 6   Alley          107 non-null    str    
 7   LotShape       1459 non-null   str    
 8   LandContour    1459 non-null   str    
 9   Utilities      1457 non-null   str    
 10  LotConfig      1459 non-null   str    
 11  LandSlope      1459 non-null   str    
 12  Neighborhood   1459 non-null   str    
 13  Condition1     1459 non-null   str    
 14  Condition2     1459 non-null   str    
 15  BldgType       1459 non-null   str    
 16  HouseStyle     1459 non-null   str    
 17  OverallQual    1459 non-null   int64  
 18

None

---test data description---


,count,mean,std,min,25%,50%,75%,max
Id,1459.0,2190.000000,421.321334,1461.0,1825.50,2190.0,2554.50,2919.0
MSSubClass,1459.0,57.378341,42.746880,20.0,20.00,50.0,70.00,190.0
LotFrontage,1232.0,68.580357,22.376841,21.0,58.00,67.0,80.00,200.0
LotArea,1459.0,9819.161069,4955.517327,1470.0,7391.00,9399.0,11517.50,56600.0
OverallQual,1459.0,6.078821,1.436812,1.0,5.00,6.0,7.00,10.0
OverallCond,1459.0,5.553804,1.113740,1.0,5.00,5.0,6.00,9.0
YearBuilt,1459.0,1971.357779,30.390071,1879.0,1953.00,1973.0,2001.00,2010.0
YearRemodAdd,1459.0,1983.662783,21.130467,1950.0,1963.00,1992.0,2004.00,2010.0
MasVnrArea,1444.0,100.709141,177.625900,0.0,0.00,0.0,164.00,1290.0
BsmtFinSF1,1458.0,439.203704,455.268042,0.0,0.00,350.5,753.50,4010.0


**2--Renaming the columns for uniformatiy**

In [291]:
#for training dataset
df_raw.columns = df_raw.columns.str.lower()   
df_raw.head()
df = df_raw.copy()
df= df.drop(columns=['id'])
#for test datasets
df_test.columns = df_test.columns.str.lower()  
df_test.head()
df_test= df_test.drop(columns=['id'])
df['saleprice']

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: saleprice, Length: 1460, dtype: int64

**As our target variable is right_skewed, so applying log1p to reduce the skewness of target variable for better model training**

In [292]:
X_train,y_train = df.drop(columns=['saleprice']),np.log1p(df['saleprice'])
X_test = df_test
print(f'shape of training dataset: {X_train.shape}')
print(f'shape of test dataset: {X_test.shape}')

shape of training dataset: (1460, 79)
shape of test dataset: (1459, 79)


**Changing the type of features based on the info they provide**

In [293]:
print('--mssubclass represents the class of sold property in real-state--\n but here its represented as an integer\n --must be transfered into nominal category, as it represents category of sold property based on style, architecture,..--')
display(df['mssubclass'].unique())
print('--mosold represents the month when the property was sold--\n which is represented as an integer\n --must be transfered into nominal category, as it represents category of month--')
display(df['mosold'].unique())
def change_datatype(df):
    df['mssubclass']=df['mssubclass'].astype(str)  #converting the type of mssubclass into string for nominal category
    df['mosold']=df['mosold'].astype(str)   #same here for month when the house was sold
    return df    

--mssubclass represents the class of sold property in real-state--
 but here its represented as an integer
 --must be transfered into nominal category, as it represents category of sold property based on style, architecture,..--


array([ 60,  20,  70,  50, 190,  45,  90, 120,  30,  85,  80, 160,  75,
       180,  40])

--mosold represents the month when the property was sold--
 which is represented as an integer
 --must be transfered into nominal category, as it represents category of month--


array([ 2,  5,  9, 12, 10,  8, 11,  4,  1,  7,  3,  6])

**1. Classification of datas based on datatypes**

In [294]:
def classify(df):
    target = 'saleprice'
    numerical = [feature for feature in df.select_dtypes(include=np.number).columns if feature!=target]
    categorical = [feature for feature in df.select_dtypes(exclude=np.number).columns if feature!=target]
    continuous = [feature for feature in numerical if df[feature].nunique() >20 if feature!=target] 
    discrete = [feature for feature in numerical if df[feature].nunique() <=20 if feature!=target]
    return numerical, categorical, continuous, discrete
   

**2. Checking the null values and duplicate values**

In [295]:
data_report = pd.DataFrame(
    {
        'null_count':df.isna().sum(),  #counting the total number of null values in each feature
        'null_percentage':(df.isna().mean() * 100).round(2)  #and converting them into percentage based on whole training examples
    }
).query('null_count>0').sort_values('null_percentage',ascending=False) #only selecting the datas having null values greater than 0 and sorting them in descending order based on null percentage
print('---The report on null values and null percentage based on every features having null values---')
display(data_report)
print(f'The features having duplicate values: {df.duplicated().sum()}')

---The report on null values and null percentage based on every features having null values---


,null_count,null_percentage
poolqc,1453,99.52
miscfeature,1406,96.30
alley,1369,93.77
fence,1179,80.75
masvnrtype,872,59.73
fireplacequ,690,47.26
lotfrontage,259,17.74
garagetype,81,5.55
garageyrblt,81,5.55
garagefinish,81,5.55


The features having duplicate values: 0


**2.1 Filling null values**

**In this ames house_price prediction datasets, most of the null values in the dataset represents that the house doesnot has this particular feature, so rather than dropping the null values feature or filling them using median or mode, we fill them with None or 0 based on the type of feature, to let the model know that the house doesnot has this particular feature**

**2.2 Every house built has a lotfrontage, so null value of this feature is actually a data_entry mistake and, as the lotfrontage of the houses in the same neighbourhood is almost identical , so null values in this feature can be filled using the median based on the neighbourhood's lotfrontage**

**2.3 Every house built has electricity facility, so null values in them is actually data entry default, so we must fill them based on freqeuntly occuring value**



In [296]:
def fill_na(df):
    df['lotfrontage'] =df.groupby('neighborhood')['lotfrontage'].transform(lambda x:x.fillna(x.median())) #filling the null values of the lotfrontage using the median of the lotfrontage based on the group
    
    df['electrical'] = df['electrical'].fillna(df['electrical'].mode()[0])
    none_features = ['poolqc', 'miscfeature', 'alley', 'fence', 'masvnrtype',
                     'fireplacequ', 'garagetype', 'garagefinish', 'garagequal',
                     'garagecond', 'bsmtexposure', 'bsmtfintype2', 'bsmtqual', 
                     'bsmtcond', 'bsmtfintype1']
    zero_features = ['masvnrarea', 'garageyrblt']
    for feature in none_features:
        df[feature] = df[feature].fillna('None')
    for feature in zero_features:
        df[feature] = df[feature].fillna(0)
    return df


**3.Initial Feature Selection**

**3.1 Checking the constant features or near constant features** 

In [303]:
def drop_constant_numerical_features(df):
    numerical, categorical, continuous, discrete = classify(df)
    vt = VarianceThreshold(0.01)
    vt.fit(df[numerical])  #checking the constant features of numerical features based on threshold 0.01
    variance_result = vt.get_support()
    constant_num_features = [col for col,s in zip(df[numerical],variance_result) if not s]
    print(f'The numerical features having constant values or near constant values: {constant_num_features}')
    df = df.drop(columns = constant_num_features)
    return df
    

def drop_constant_categorical_features(df,y):
    numerical, categorical, continuous, discrete = classify(df)
    ordinal_categories = {
    'lotshape':      ['IR3', 'IR2', 'IR1', 'Reg'],
    'exterqual':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'extercond':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtqual':      ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtcond':      ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtexposure':  ['None', 'No', 'Mn', 'Av', 'Gd'],
    'bsmtfintype1':  ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'bsmtfintype2':  ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'heatingqc':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'kitchenqual':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'functional':    ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'fireplacequ':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garagefinish':  ['None', 'Unf', 'RFn', 'Fin'],
    'garagequal':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garagecond':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'paveddrive':    ['N', 'P', 'Y'],
    'poolqc':        ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'fence':         ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'electrical':    ['Mix', 'FuseP', 'FuseF', 'FuseA', 'SBrkr']}

    nominal_categories = [
    'mssubclass', 'mszoning', 'alley', 'landcontour', 'lotconfig',
    'neighborhood', 'condition1', 'condition2', 'bldgtype', 'housestyle',
    'roofstyle', 'roofmatl', 'exterior1st', 'exterior2nd', 'masvnrtype',
    'foundation', 'heating', 'centralair', 'garagetype', 'miscfeature',
    'saletype', 'salecondition', 'mosold',
]
    
    
    constant_cat_features = [feature for feature in categorical if df[feature].value_counts(normalize=True).iloc[0]>0.99]  #if the most frequently occuring category of this current feature is seen in almost 99% of training data, then this feature is considered as constant categorical feature 
    mi_storage = {}
    for feature in constant_cat_features:
        if feature in ordinal_categories:
            mapping = {k:i for i,k in enumerate(ordinal_categories[feature])} #creating the dictionary which shows the category as key and its corresponding index as value, here index represents the position of the category in the given order of current feature
            x = df[feature].map(mapping).to_frame(name=feature) 
        else:
            x = df[feature].astype('category').cat.codes.to_frame(name=feature)   #converting the current categorical features value into cat codes
        mi = mutual_info_regression(x,y)  #calculating the statistical relationship between categories and the target variable
        mi_storage[feature] = mi
    features_to_drop = [feature for feature,mi_value in mi_storage.items() if mi_value<0.01]
    
    df = df.drop(columns = features_to_drop)   
    return df
df = drop_constant_categorical_features(X_train,y_train)
df.columns
    
    
    


Index(['mssubclass', 'mszoning', 'lotfrontage', 'lotarea', 'alley', 'lotshape',
       'landcontour', 'lotconfig', 'landslope', 'neighborhood', 'condition1',
       'condition2', 'bldgtype', 'housestyle', 'overallqual', 'overallcond',
       'yearbuilt', 'yearremodadd', 'roofstyle', 'roofmatl', 'exterior1st',
       'exterior2nd', 'masvnrtype', 'masvnrarea', 'exterqual', 'extercond',
       'foundation', 'bsmtqual', 'bsmtcond', 'bsmtexposure', 'bsmtfintype1',
       'bsmtfinsf1', 'bsmtfintype2', 'bsmtfinsf2', 'bsmtunfsf', 'totalbsmtsf',
       'heating', 'heatingqc', 'centralair', 'electrical', '1stflrsf',
       '2ndflrsf', 'lowqualfinsf', 'grlivarea', 'bsmtfullbath', 'bsmthalfbath',
       'fullbath', 'halfbath', 'bedroomabvgr', 'kitchenabvgr', 'kitchenqual',
       'totrmsabvgrd', 'functional', 'fireplaces', 'fireplacequ', 'garagetype',
       'garageyrblt', 'garagefinish', 'garagecars', 'garagearea', 'garagequal',
       'garagecond', 'paveddrive', 'wooddecksf', 'openporchsf',
    

**4. Initial Feature Engineering**


**4.1 Creating new features**

In [255]:
def create_age_features(df):
    df['house_age'] = df['yrsold'] - df['yearbuilt']  #creating the age of house
    df['garage_age'] = np.where(df['garageyrblt']!=0,df['yrsold'] - df['garageyrblt'],-1) #creating garage_age, if the garageyrblt is 0, then it means the house didn't had any garage, so we are filling garage age wth -1 in such case
    df['remodeled_age'] = df['yrsold'] - df['yearremodadd'] #calculating the age of house since it was remodeled
    df['has_pool'] = (df['poolarea'] > 0).astype(int) #Has pool?
    df['has_garage'] = (df['garagearea'] > 0).astype(int) #Has garage?
    df['has_fireplace'] = (df['fireplaces'] > 0).astype(int) #Has fireplace?
    df['has_basement'] = (df['totalbsmtsf'] > 0).astype(int)# Has basement?
    df['has_2nd_floor'] = (df['2ndflrsf'] > 0).astype(int) #Has 2nd floor?
    df['has_masonry'] = (df['masvnrarea'] > 0).astype(int) #Has masonry veneer?
    return df



**Dropping the date features, after the creation of related age features, as age fatures are more informative than the data features**

In [256]:
def drop_date_features(df):
    df = df.drop(columns=['yearbuilt','yrsold','garageyrblt','yearremodadd'])
    return df


**Again classifying the datasets after addition of new features**

In [210]:
numerical, categorical, continuous, discrete = classify(df)   

**5. Feature Selection** 

**a) Numerical Feature Selection**

**5.1 Checking correlation of numerical features with the target variable, and extracting the important features**

In [211]:
target = 'saleprice'
corr_report = []
for feature in numerical:
    corr = df[feature].corr(df[target])  #finding the correlation of all the numeric features with the target variable
    corr_report.append({
        'feature':feature,
        'corr_with_target':corr
    })
df_cor_report = pd.DataFrame(corr_report)
imp_num_features_report = df_cor_report[df_cor_report['corr_with_target'].abs()>0.1].sort_values('corr_with_target',ascending=False,key = lambda x:x.abs())
imp_num_features = imp_num_features_report['feature']
imp_num_features_report

,feature,corr_with_target
2,overallqual,0.817185
12,grlivarea,0.700927
21,garagecars,0.680625
22,garagearea,0.650888
8,totalbsmtsf,0.612134
9,1stflrsf,0.596981
15,fullbath,0.594771
30,house_age,-0.587290
32,remodeled_age,-0.568136
19,totrmsabvgrd,0.534422


**Correlation matrix of imporatant numerical features**

In [212]:
imp_corr_matrix = df[imp_num_features].corr()  #finding the correlation of each important numerical features with eachother
for i in range(len(imp_corr_matrix.columns)):
    for j in range(i):
        r = imp_corr_matrix.iloc[i,j]  #extracting the correlation value
        ci = imp_corr_matrix.columns[i]
        cj = imp_corr_matrix.columns[j]
        if ci!=cj and abs(r)>0.75 and ci!=target and cj!=target:
            print(f'{ci}<-->{cj} having r = {r:.2f}')

garagearea<-->garagecars having r = 0.88
1stflrsf<-->totalbsmtsf having r = 0.82
totrmsabvgrd<-->grlivarea having r = 0.83
fireplaces<-->has_fireplace having r = 0.90
has_2nd_floor<-->2ndflrsf having r = 0.91


**Now, based on the higly correlated numerical features, and based on the report of corr with target, we can drop the weaker numerical feature from the above pairs**   

**So, dropping the columns garagearea, grlivarea, 1stflrsf,  totrmsabvgrd , fireplaces**

In [213]:
features_to_drop = ['garagearea', 'totrmsabvgrd', '1stflrsf', 'fireplaces','has_2nd_floor']
df = df.drop(columns = features_to_drop)


**Checking the multi_collinearity among the important numerical features**

In [214]:
vif_features = [feature for feature in imp_num_features if feature not in features_to_drop] #as we have already dropped the weaker numerical features among the important numerical features
variables = df[vif_features].dropna()
vif = pd.DataFrame({
    'features':variables.columns,
    'vif_value':[variance_inflation_factor(variables,i) for i in range(variables.shape[1])],
    'corr_with_target':df[vif_features].corrwith(df['saleprice'])  #finding the correlation with the target variable
}).query('corr_with_target>0.1').sort_values('vif_value',ascending=False)
vif

,features,vif_value,corr_with_target
grlivarea,grlivarea,96.949124,0.700927
totalbsmtsf,totalbsmtsf,88.340472,0.612134
has_basement,has_basement,51.538419,0.199634
overallqual,overallqual,50.854142,0.817185
has_garage,has_garage,47.567345,0.322998
fullbath,fullbath,25.446196,0.594771
bsmtunfsf,bsmtunfsf,23.500794,0.221985
bedroomabvgr,bedroomabvgr,23.450915,0.209043
garagecars,garagecars,20.365895,0.680625
bsmtfinsf1,bsmtfinsf1,17.960404,0.372023


**From the above vif report and comparing the correlation with the target, we have decided to drop the features 'has_basement','bedroomabvgr','bsmtunfsf' which have a very high vif_value and very low correlation value with target among the above important numerical features**

In [215]:
df = df.drop(columns = ['has_basement','bedroomabvgr','bsmtunfsf'])

In [216]:

numerical, categorical, continuous, discrete = classify(df)   


In [217]:
df.columns

Index(['mssubclass', 'mszoning', 'lotfrontage', 'lotarea', 'alley', 'lotshape',
       'landcontour', 'utilities', 'lotconfig', 'landslope', 'neighborhood',
       'condition1', 'condition2', 'bldgtype', 'housestyle', 'overallqual',
       'overallcond', 'roofstyle', 'roofmatl', 'exterior1st', 'exterior2nd',
       'masvnrtype', 'masvnrarea', 'exterqual', 'extercond', 'foundation',
       'bsmtqual', 'bsmtcond', 'bsmtexposure', 'bsmtfintype1', 'bsmtfinsf1',
       'bsmtfintype2', 'bsmtfinsf2', 'totalbsmtsf', 'heating', 'heatingqc',
       'centralair', 'electrical', '2ndflrsf', 'lowqualfinsf', 'grlivarea',
       'bsmtfullbath', 'bsmthalfbath', 'fullbath', 'halfbath', 'kitchenabvgr',
       'kitchenqual', 'functional', 'fireplacequ', 'garagetype',
       'garagefinish', 'garagecars', 'garagequal', 'garagecond', 'paveddrive',
       'wooddecksf', 'openporchsf', 'enclosedporch', '3ssnporch',
       'screenporch', 'poolarea', 'poolqc', 'fence', 'miscfeature', 'miscval',
       'mosold', '

**b) Categorical Feature Selection**

**ANOVA TEST for categorical features**

In [218]:
anova_report = []
for feature in categorical:
    groups = [group['saleprice'].values for _,group in df.groupby(feature)]  #extracting the values of the saleprice based on different categories of current feature
    f_stats,p_value = stats.f_oneway(*groups)  #anova test of different saleprice values based on each  categories of current feature
    anova_report.append({
        'feature':feature,
        'f_stats':f_stats,
        'p_value':format(p_value,'.250f')
    })
total_result_cat = pd.DataFrame(anova_report).sort_values('p_value')
total_result_cat

    

,feature,f_stats,p_value
8,neighborhood,79.520526,0.00000000000000000000000000000000000000000000...
18,exterqual,415.304259,0.00000000000000000000000000000000000000000000...
21,bsmtqual,300.392915,0.00000000000000000000000000000000000000000000...
30,kitchenqual,393.320922,0.00000000000000000000000000000000000000000000...
34,garagefinish,298.769591,0.00000000000000000000000000000000000000000000...
33,garagetype,121.796238,0.00000000000000000000000000000000000000000000...
0,mssubclass,50.866063,0.00000000000000000000000000000000000000000000...
32,fireplacequ,131.198588,0.00000000000000000000000000000000000000000000...
20,foundation,126.806779,0.00000000000000000000000000000000000000000000...
27,heatingqc,110.820423,0.00000000000000000000000000000000000000000000...


**From the above report, we can observe that features such as LandSlope, MoSold, and Utilities have very low F-statistics and p-values greater than 0.05, indicating that they have a weak statistical relationship with the target variable and are not significant predictors. Therefore, we are dropping these features.**

In [219]:
df = df.drop(columns = ['landslope','mosold','utilities'])
df.columns

Index(['mssubclass', 'mszoning', 'lotfrontage', 'lotarea', 'alley', 'lotshape',
       'landcontour', 'lotconfig', 'neighborhood', 'condition1', 'condition2',
       'bldgtype', 'housestyle', 'overallqual', 'overallcond', 'roofstyle',
       'roofmatl', 'exterior1st', 'exterior2nd', 'masvnrtype', 'masvnrarea',
       'exterqual', 'extercond', 'foundation', 'bsmtqual', 'bsmtcond',
       'bsmtexposure', 'bsmtfintype1', 'bsmtfinsf1', 'bsmtfintype2',
       'bsmtfinsf2', 'totalbsmtsf', 'heating', 'heatingqc', 'centralair',
       'electrical', '2ndflrsf', 'lowqualfinsf', 'grlivarea', 'bsmtfullbath',
       'bsmthalfbath', 'fullbath', 'halfbath', 'kitchenabvgr', 'kitchenqual',
       'functional', 'fireplacequ', 'garagetype', 'garagefinish', 'garagecars',
       'garagequal', 'garagecond', 'paveddrive', 'wooddecksf', 'openporchsf',
       'enclosedporch', '3ssnporch', 'screenporch', 'poolarea', 'poolqc',
       'fence', 'miscfeature', 'miscval', 'saletype', 'salecondition',
       'salepri

In [220]:

numerical, categorical, continuous, discrete = classify(df)   

**6. Feature Engineering**

**6.1 Grouping of rare categories**

In [221]:
def rare_categories(df,data_type):
    rare_cat_dict = {}
    for feature in data_type:
        frequency = df[feature].value_counts(normalize=True)  #couting the frequency of every category in this current feature
        rare_categories = frequency[frequency<0.01].index.tolist()   #converting the result of rare categories into list, here rare categories means the categories which appear in less than 1 % of the total training datasets
        if rare_categories:
            rare_cat_dict[feature] = rare_categories
    return rare_cat_dict  
rare_cat_dict =   rare_categories(df,categorical)  

def replace_rare_categories(df,rare_cat_dict): #this function is used for replacing the rare categories with name 'other' in each categorical feauture
    for feature,categories in rare_cat_dict.items():
        df[feature] = df[feature].apply(lambda x: 'other' if x in categories else x)  #replacing the rare categories with name 'other'
    return df
df = replace_rare_categories(df,rare_cat_dict)    

#checking the replacement of rare categories
column = 'salecondition' 
df[column].value_counts(normalize=True)

        

salecondition
Normal     0.820548
Partial    0.085616
Abnorml    0.069178
Family     0.013699
other      0.010959
Name: proportion, dtype: float64

**6.2 Feature Encoding**


a) One_hot encoding for nominal categories

In [222]:
nominal_categories = [
    'mssubclass', 'mszoning', 'alley', 'landcontour', 'lotconfig',
    'neighborhood', 'condition1', 'condition2', 'bldgtype', 'housestyle',
    'roofstyle', 'roofmatl', 'exterior1st', 'exterior2nd', 'masvnrtype',
    'foundation', 'heating', 'centralair', 'garagetype', 'miscfeature',
     'saletype', 'salecondition'
]

In [223]:
ordinal_categories = [
    'lotshape', 'exterqual', 'extercond', 'bsmtqual', 'bsmtcond',
    'bsmtexposure', 'bsmtfintype1', 'bsmtfintype2', 'heatingqc', 'kitchenqual',
    'functional', 'fireplacequ', 'garagefinish', 'garagequal', 'garagecond',
    'paveddrive', 'poolqc', 'fence', 'electrical'
]

In [224]:
print('length of nominal categories:', len(nominal_categories))
print('length of ordinal categories:', len(ordinal_categories))
print('length of categorical features:', len(categorical))

length of nominal categories: 22
length of ordinal categories: 19
length of categorical features: 41


In [225]:
nominal_encoded = pd.get_dummies(data=df[nominal_categories],columns=nominal_categories,drop_first=True,dtype=int)
nominal_encoded.head()


,mssubclass_160,mssubclass_190,mssubclass_20,mssubclass_30,mssubclass_50,mssubclass_60,mssubclass_70,mssubclass_75,mssubclass_80,mssubclass_85,...,garagetype_other,miscfeature_Shed,miscfeature_other,saletype_New,saletype_WD,saletype_other,salecondition_Family,salecondition_Normal,salecondition_Partial,salecondition_other
0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
3,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0


In [226]:
ordinal_mapping = {
    'lotshape': ['other', 'IR2', 'IR1', 'Reg'],                          # IR3 replaced by 'other'

    'exterqual': ['other', 'Fa', 'TA', 'Gd', 'Ex'],                      # Po replaced by 'other'

    'extercond': ['other', 'Fa', 'TA', 'Gd', 'Ex'],                      # Po replaced by 'other'

    'bsmtqual': ['None', 'Fa', 'TA', 'Gd', 'Ex'],                        # None = no basement

    'bsmtcond': ['None', 'other', 'Fa', 'TA', 'Gd'],                     # None = no basement

    'bsmtexposure': ['None', 'No', 'Mn', 'Av', 'Gd'],                    # None = no basement

    'bsmtfintype1': ['None', 'Unf','LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'], # None = no basement

    'bsmtfintype2': ['None','Unf' ,'other', 'LwQ', 'Rec', 'BLQ', 'ALQ' ],# None = no basement

    'heatingqc': ['other', 'Fa', 'TA', 'Gd', 'Ex'],                      # Po replaced by 'other'

    'kitchenqual': ['Fa', 'TA', 'Gd', 'Ex'],                             # Po not present

    'functional': ['other', 'Mod', 'Min2', 'Min1', 'Typ'],               # Sal/Sev/Maj replaced by 'other'

    'fireplacequ': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],               # None = no fireplace

    'garagefinish': ['None', 'Unf', 'RFn', 'Fin'],                       # None = no garage

    'garagequal': ['None', 'other', 'Fa', 'TA'],                         # None = no garage

    'garagecond': ['None', 'other', 'Fa', 'TA'],                         # None = no garage

    'paveddrive': ['N', 'P', 'Y'],

    'poolqc': ['None', 'other'],                                          # almost all None

    'fence': ['None', 'other', 'GdWo', 'MnPrv', 'GdPrv'],               # MnWw replaced by 'other'

    'electrical': ['other', 'FuseF', 'FuseA', 'SBrkr']                   # Mix/FuseP replaced by 'other'
}

In [227]:
ordinal_encoded = df[ordinal_categories].copy()
for key,value in ordinal_mapping.items():
    mapping_dict = {k:i for i,k in enumerate(value)}  #here we are creating a dict of name of category as key and its index as value, for the mapping of order
    ordinal_encoded[key] = df[key].map(mapping_dict) #here we are assigning the numerical value or power value based on the index of the key in the ordinal_mapping , which is stored as a dict inside the loop
ordinal_encoded.head()

,lotshape,exterqual,extercond,bsmtqual,bsmtcond,bsmtexposure,bsmtfintype1,bsmtfintype2,heatingqc,kitchenqual,functional,fireplacequ,garagefinish,garagequal,garagecond,paveddrive,poolqc,fence,electrical
0,3,3,2,3,3,1,6,6,4,2,4,0,2,3,3,2,0,0,3
1,3,2,2,3,3,4,4,6,4,1,4,3,2,3,3,2,0,0,3
2,2,3,2,3,3,2,6,6,4,2,4,3,2,3,3,2,0,0,3
3,2,2,2,2,4,1,4,6,3,2,4,4,1,3,3,2,0,0,3
4,2,3,2,3,3,3,6,6,4,2,4,3,2,3,3,2,0,0,3


In [228]:
final_encoded_cat = pd.concat([nominal_encoded,ordinal_encoded],axis=1) 
final_encoded_cat.head()

,mssubclass_160,mssubclass_190,mssubclass_20,mssubclass_30,mssubclass_50,mssubclass_60,mssubclass_70,mssubclass_75,mssubclass_80,mssubclass_85,...,kitchenqual,functional,fireplacequ,garagefinish,garagequal,garagecond,paveddrive,poolqc,fence,electrical
0,0,0,0,0,0,1,0,0,0,0,...,2,4,0,2,3,3,2,0,0,3
1,0,0,1,0,0,0,0,0,0,0,...,1,4,3,2,3,3,2,0,0,3
2,0,0,0,0,0,1,0,0,0,0,...,2,4,3,2,3,3,2,0,0,3
3,0,0,0,0,0,0,1,0,0,0,...,2,4,4,1,3,3,2,0,0,3
4,0,0,0,0,0,1,0,0,0,0,...,2,4,3,2,3,3,2,0,0,3


**Concating the final numerical and categorical features**

In [229]:
numerical_df = df[numerical]
final_training_features = pd.concat([final_encoded_cat,numerical_df],axis=1)
final_training_features.head()

,mssubclass_160,mssubclass_190,mssubclass_20,mssubclass_30,mssubclass_50,mssubclass_60,mssubclass_70,mssubclass_75,mssubclass_80,mssubclass_85,...,screenporch,poolarea,miscval,house_age,garage_age,remodeled_age,has_pool,has_garage,has_fireplace,has_masonry
0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,5,5.0,5,0,1,0,1
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,31,31.0,31,0,1,1,0
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,7,7.0,6,0,1,1,1
3,0,0,0,0,0,0,1,0,0,0,...,0,0,0,91,8.0,36,0,1,1,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,8,8.0,8,0,1,1,1


**6.3 Splitting into training, cross_validation and test dataset**

In [230]:
X = np.array(final_training_features)
y = np.array(df['saleprice'])
print('shape of X:', X.shape)
print('shape of y:', y.shape)
X_train,X_,y_train,y_ = train_test_split(X,y,test_size = 0.4,random_state=55)
X_cv,X_test,y_cv,y_test = train_test_split(X_,y_,test_size = 0.5,random_state=55)
del X_,y_
print('shape of train dataset:', X_train.shape)
print('shape of cross_validation dataset:', X_cv.shape)
print('shape of test dataset:', X_test.shape)

shape of X: (1460, 159)
shape of y: (1460,)
shape of train dataset: (876, 159)
shape of cross_validation dataset: (292, 159)
shape of test dataset: (292, 159)


**6.4 Feature scaling**

In [231]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_cv_scaled = sc.transform(X_cv)
X_test_scaled = sc.transform(X_test)

**7 Training the model**

In [232]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
lr

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


**prediction done on training dataset**

In [233]:
predicted_y_train = lr.predict(X_train_scaled)
print('The first five predicted training saleprice is:', predicted_y_train[:5])
print('The actual first five training saleprice is:', y_train[:5])


The first five predicted training saleprice is: [12.4912457  11.79985674 11.77441284 11.95273565 11.98016298]
The actual first five training saleprice is: [12.52453001 11.93164238 11.67420212 11.94039991 11.89136875]


In [234]:
predicted_y_cv = lr.predict(X_cv_scaled)
print('The first five predicted cross_validation saleprice is:', predicted_y_cv[:5])
print('The actual first five cross_validation saleprice is:', y_cv[:5])

The first five predicted cross_validation saleprice is: [11.73882032 12.29480887 12.04175203 12.15070826 11.88586805]
The actual first five cross_validation saleprice is: [11.87688084 12.39255636 12.03469696 12.0435596  11.90834697]


In [235]:
predicted_y_test = lr.predict(X_test_scaled)
print('The first five predicted test saleprice is:', predicted_y_cv[:5])
print('The actual first five test saleprice is:', y_cv[:5])

The first five predicted test saleprice is: [11.73882032 12.29480887 12.04175203 12.15070826 11.88586805]
The actual first five test saleprice is: [11.87688084 12.39255636 12.03469696 12.0435596  11.90834697]


**Cost_function of prediction of different datasets**

In [236]:
y_pred_train_original = np.expm1(predicted_y_train)
y_train_original = np.expm1(y_train)
cost_train = mean_absolute_error(y_train_original,y_pred_train_original)
print('The cost function of training dataset is:', cost_train)
y_pred_cv_original = np.expm1(predicted_y_cv)
y_cv_original = np.expm1(y_cv)
cost_cv = mean_absolute_error(y_cv_original,y_pred_cv_original)
print('The cost function of cross_validation dataset is:', cost_cv)
y_pred_test_original = np.expm1(predicted_y_test)
y_test_original = np.expm1(y_test)
cost_test = mean_absolute_error(y_test_original,y_pred_test_original)
relative_error =  (cost_test / y_test_original.mean()) * 100  #here we are calculating by how much percent our predicted saleprice was off from the actual saleprice amount
print(f'The relative error is: {relative_error}')


The cost function of training dataset is: 11488.62204193856
The cost function of cross_validation dataset is: 15325.033164465274
The relative error is: 14.952750823736139


**Predicting with multiple models**

In [237]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost' : XGBRegressor(n_estimators=200,learning_rate=0.05,verbosity=1,random_state = 42,early_stopping_rounds = 10)
}
results = []
for name,model in models.items():
    if name == 'Linear Regression':
        model.fit(X_train_scaled,y_train)
        y_predicted = model.predict(X_test_scaled)  #prediction done through test dataset
    elif name == 'XGBoost':  #as for the XGboost model , evaluation dataset is required
        model.fit(X_train,y_train,eval_set = [(X_test,y_test)])  
        y_predicted = model.predict(X_test)
    else:  #if the model is not linear regression and not XGBoost, instead a tree ensembles then we use X_test, not scaled X_test
        model.fit(X_train,y_train) 
        y_predicted = model.predict(X_test)
    y_pred_original = np.expm1(y_predicted)
    y_test_original = np.expm1(y_test)    
    mae  = mean_absolute_error(y_test_original, y_pred_original)
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
    r2   = r2_score(y_test_original, y_pred_original)    
    results.append({'name':name,'model':model,'MAE':mae,'RMSE':rmse,'R2':r2})
for model in results:
    print(f'Model: {model['name']}, MAE: ${model['MAE']}, RMSE: {model['RMSE']}, R2: {model['R2']}  ')
result_df = pd.DataFrame([{'model':r['name'], 'MAE':r['MAE'], 'RMSE':r['RMSE'], 'R2':r['R2']} for r in results]) 


[0]	validation_0-rmse:0.38536
[1]	validation_0-rmse:0.37085
[2]	validation_0-rmse:0.35736
[3]	validation_0-rmse:0.34432
[4]	validation_0-rmse:0.33215
[5]	validation_0-rmse:0.32076
[6]	validation_0-rmse:0.31016
[7]	validation_0-rmse:0.30034
[8]	validation_0-rmse:0.29050
[9]	validation_0-rmse:0.28154
[10]	validation_0-rmse:0.27329
[11]	validation_0-rmse:0.26516
[12]	validation_0-rmse:0.25778
[13]	validation_0-rmse:0.25120
[14]	validation_0-rmse:0.24466
[15]	validation_0-rmse:0.23845
[16]	validation_0-rmse:0.23246
[17]	validation_0-rmse:0.22740
[18]	validation_0-rmse:0.22238
[19]	validation_0-rmse:0.21765
[20]	validation_0-rmse:0.21373
[21]	validation_0-rmse:0.20997
[22]	validation_0-rmse:0.20615
[23]	validation_0-rmse:0.20278
[24]	validation_0-rmse:0.19992
[25]	validation_0-rmse:0.19689
[26]	validation_0-rmse:0.19408
[27]	validation_0-rmse:0.19160
[28]	validation_0-rmse:0.18952
[29]	validation_0-rmse:0.18757
[30]	validation_0-rmse:0.18610
[31]	validation_0-rmse:0.18484
[32]	validation_0-

In [238]:
result_df

,model,MAE,RMSE,R2
0,Linear Regression,26558.867084,165869.767933,-3.375487
1,Decision Tree,28653.330332,61153.772197,0.405245
2,Random Forest,19486.874265,40801.015207,0.735251
3,Gradient Boosting,18941.114596,43260.741559,0.702368
4,XGBoost,19807.087570,42831.437747,0.708246
